### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import sys
sys.path.append('./utils')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from siglip_class import SVGMetricEvaluator

In [3]:
from unsloth import FastLanguageModel
import torch
import random
import numpy as np
import multiprocessing as mp

mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

/tmp/ipykernel_14757/1875917155.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-22 22:58:58 [__init__.py:239] Automatically detected platform cuda.


In [4]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc


class Model:
    
    def __init__(self):

        self.model_path = "./lora/Qwen25_7B_Instruct_lora_fp16_r256_s2000_i1000_msl2048"

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
        model_name = self.model_path, # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        dtype = (None),
        load_in_4bit = False,
        )
        FastLanguageModel.for_inference(self.model).to('cuda') # Enable native 2x faster inference

       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     
    
    def get_response(self, description):

        #alpaca prompt
           
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.
            
                ### Instruction:
                Generate a SVG code for the given input:
            
                ### Input:             
                {}
            
                ### Response:
                """
        
        formatted_input = alpaca_prompt.format(description)
        inputs = self.tokenizer([formatted_input], return_tensors="pt").to('cuda')
        outputs = self.model.generate(**inputs, max_new_tokens=1024, use_cache=True)
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
    
    def predict(self, description: str, max_new_tokens=1024) -> str:
        output_decoded = self.get_response(description)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)
        

In [5]:
model=Model()

==((====))==  Unsloth 2025.3.18: Fast Qwen2 patching. Transformers: 4.47.1. vLLM: 0.8.2.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.695 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
model.predict('sun rising in the east')

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 150" width="400" height="300"><defs><radialGradient id="sunGradient" cx="0.5" cy="0.5" r="0.5"><stop offset="0%" stop-color="#FFD700"/><stop offset="50%" stop-color="#FFA500"/><stop offset="100%" stop-color="#FF4500"/></radialGradient><linearGradient id="skyGradient" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="#1F5FFF"/><stop offset="50%" stop-color="#64B5FF"/><stop offset="100%" stop-color="#E3F2FD"/></linearGradient><linearGradient id="horizonGradient" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="#FFD700"/><stop offset="100%" stop-color="#FF8C00"/></linearGradient></defs><rect x="0" y="0" width="200" height="100" fill="url(#skyGradient)"/><polygon points="0,100 30,80 60,110 99,930 130,85 160,110 190,80 200,100" fill="#FF4500" opacity="0.3"/><circle cx="100" cy="50" r="25" fill="url(#sunGradient)"/><g stroke="#FFD700" stroke-width="2" stroke-linecap="round"><line x1="100" y1="28" x2="100" y2="0"/><lin

In [7]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)

(75, 7)


,description,gpt_svg,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [9]:
from tqdm import tqdm
tqdm.pandas()
df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [ ]:
df.to_csv('tmp.csv',index=False)

In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import pandas as pd

# # Assume `df` and `model` are defined
# descriptions = df['description'].tolist()
# results = []

# # Number of threads (tune based on GPU usage & RAM availability)
# NUM_THREADS = 8

# with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
#     future_to_desc = {executor.submit(model.predict, desc): desc for desc in descriptions}
    
#     for future in tqdm(as_completed(future_to_desc), total=len(descriptions)):
#         try:
#             result = future.result()
#         except Exception as e:
#             result = model.default_svg  # fallback or error logging
#         results.append(result)

# # Add results back to dataframe
# df['svg_3'] = results


In [ ]:
#model.close_model()

In [ ]:
import pandas as pd
df=pd.read_csv('tmp.csv')
df.head(2)

In [ ]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

In [ ]:
df['svg_score_3'].mean()